#  Notebook 04 — Componente LLM: Explicabilidad con Groq + SHAP
## EnergyForecast · IA EAFIT 2026-1

**Prerequisito:** Haber ejecutado los notebooks 01, 02 y 03.

**Objetivo:** El LLM recibe las predicciones del modelo XGBoost junto con los
SHAP values de las features más importantes, y genera una **explicación en lenguaje
natural** comprensible para operadores sin formación técnica en ML.

**Modelo LLM:** `llama-3.1-8b-instant` vía **Groq API** (gratuita)

**Estrategia de prompting:** Few-shot con chain-of-thought

In [ ]:
# ── Instalación ───────────────────────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'groq', 'xgboost', 'shap', 'joblib', '-q'])
print('✅ Dependencias OK')

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os, json, time
import pandas as pd
import numpy as np
import joblib
import shap
from groq import Groq


GROQ_API_KEY =  os.environ.get('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')


client = Groq(api_key=GROQ_API_KEY)
LLM_MODEL = 'llama-3.1-8b-instant'   # modelo gratuito y rápido en Groq

print(f'✅ Groq client inicializado | Modelo: {LLM_MODEL}')

## 1. Cargar modelo y datos

In [ ]:
# ── Cargar modelo XGBoost y SHAP explainer ────────────────────────────────────
xgb_model = joblib.load('models/checkpoints/xgb_energy.pkl')
explainer  = shap.TreeExplainer(xgb_model)

with open('data/processed/feature_cols.json') as f:
    FEATURE_COLS = json.load(f)

X_test = pd.read_csv('data/processed/X_test_xgb.csv', index_col='Datetime', parse_dates=True)
y_test = pd.read_csv('data/processed/y_test_xgb.csv', index_col='Datetime', parse_dates=True)['MW']

# Cargar ejemplo pre-guardado del notebook 03
with open('data/processed/example_prediction.json') as f:
    example = json.load(f)

print('✅ Modelo y datos cargados')
print(f'   Test set: {len(X_test):,} registros')

## 2. Sistema de prompting few-shot con SHAP

In [ ]:
# ── Función para construir el contexto SHAP de una predicción ─────────────────
def get_shap_context(x_row: pd.Series, feature_cols: list) -> dict:
    """
    Calcula SHAP values para una fila y devuelve el top-5 como dict.
    """
    shap_vals = explainer.shap_values(x_row[feature_cols].values.reshape(1, -1))[0]
    shap_dict = dict(zip(feature_cols, shap_vals))
    top5 = sorted(shap_dict.items(), key=lambda kv: abs(kv[1]), reverse=True)[:5]
    return {k: round(float(v), 1) for k, v in top5}


def build_prompt(pred_mw: float, timestamp: str, hour: int, month: int,
                 lag_24h: float, lag_168h: float, top5_shap: dict) -> str:
    """
    Construye el prompt few-shot para el LLM.
    Estrategia: sistema + 2 ejemplos demostrativos + pregunta real.
    """
    month_names = ['', 'enero','febrero','marzo','abril','mayo','junio',
                   'julio','agosto','septiembre','octubre','noviembre','diciembre']
    mes = month_names[month]

    # Formatear SHAP como texto legible
    shap_lines = '\n'.join(
        f'  - {feat}: {val:+.1f} MW ({"aumenta" if val > 0 else "reduce"} el consumo)'
        for feat, val in top5_shap.items()
    )

    system_prompt = """Eres un experto en sistemas eléctricos que explica predicciones \
de consumo energético a operadores sin formación técnica. \
Tu explicación debe:
1. Mencionar la predicción en MW y el intervalo de confianza aproximado (±5%).
2. Explicar las 3 razones principales del consumo usando los valores SHAP.
3. Usar lenguaje simple, sin jerga técnica de Machine Learning.
4. Ser concisa: máximo 4 oraciones.
5. Responder SIEMPRE en español."""

    few_shot_example_1 = """
--- EJEMPLO 1 ---
Predicción: 41,200 MW para las 18h del martes de julio
Consumo hace 24h (lag_24h): 40,800 MW | Consumo misma hora semana pasada (lag_168h): 41,000 MW
Factores SHAP principales:
  - hour: +2,100 MW (aumenta el consumo)
  - lag_168h: +980 MW (aumenta el consumo)
  - month: +650 MW (aumenta el consumo)

EXPLICACIÓN ESPERADA:
Se proyecta un consumo de 41,200 MW para las 6 PM de este martes \
(rango estimado: 39,100–43,300 MW). El principal impulsor es el horario \
pico vespertino, cuando hogares y comercios aumentan su demanda tras la \
jornada laboral. Adicionalmente, el consumo de la semana pasada a esta \
misma hora fue muy similar (41,000 MW), lo que confirma el patrón semanal. \
El calor de julio también contribuye con mayor uso de aire acondicionado.
"""

    few_shot_example_2 = """
--- EJEMPLO 2 ---
Predicción: 28,500 MW para las 4h del domingo de diciembre
Consumo hace 24h (lag_24h): 30,100 MW | Consumo misma hora semana pasada (lag_168h): 29,000 MW
Factores SHAP principales:
  - hour: -3,200 MW (reduce el consumo)
  - is_weekend: -1,400 MW (reduce el consumo)
  - month: +800 MW (aumenta el consumo)

EXPLICACIÓN ESPERADA:
El sistema proyecta 28,500 MW para las 4 AM del domingo \
(rango: 27,000–30,000 MW), un nivel bajo propio del horario nocturno \
cuando la mayoría de actividades están detenidas. El hecho de ser domingo \
refuerza la baja demanda al no haber actividad industrial ni comercial. \
El frío de diciembre modera parcialmente esta reducción al mantener \
encendidos los sistemas de calefacción.
"""

    user_query = f"""
--- CASO REAL ---
Predicción: {pred_mw:,.0f} MW para las {hour}h del mes de {mes} ({timestamp})
Consumo hace 24h (lag_24h): {lag_24h:,.0f} MW | Consumo misma hora semana pasada (lag_168h): {lag_168h:,.0f} MW
Factores SHAP principales:
{shap_lines}

Genera la EXPLICACIÓN para este caso:"""

    full_user = few_shot_example_1 + few_shot_example_2 + user_query

    return system_prompt, full_user


print('✅ Funciones de prompting definidas')

## 3. Función principal de explicación

In [ ]:
# ── Función de explicación completa ───────────────────────────────────────────
def explain_prediction(idx: int, verbose: bool = True) -> dict:
    """
    Para un índice en X_test:
    1. Obtiene predicción XGBoost
    2. Calcula SHAP values
    3. Llama al LLM (Groq) para generar la explicación
    4. Retorna dict con predicción, SHAP y explicación
    """
    row = X_test.iloc[idx]
    ts  = X_test.index[idx]

    # Predicción
    pred_mw = float(xgb_model.predict(row[FEATURE_COLS].values.reshape(1, -1))[0])
    real_mw = float(y_test.iloc[idx])
    error   = abs(pred_mw - real_mw) / real_mw * 100

    # SHAP
    top5_shap = get_shap_context(row, FEATURE_COLS)

    # Prompt
    system_p, user_p = build_prompt(
        pred_mw  = pred_mw,
        timestamp= str(ts),
        hour     = int(row['hour']),
        month    = int(row['month']),
        lag_24h  = float(row['lag_24h']),
        lag_168h = float(row['lag_168h']),
        top5_shap= top5_shap,
    )

    # Llamada al LLM
    t0 = time.time()
    completion = client.chat.completions.create(
        model     = LLM_MODEL,
        messages  = [
            {'role': 'system', 'content': system_p},
            {'role': 'user',   'content': user_p},
        ],
        max_tokens  = 350,
        temperature = 0.3,    # baja temperatura = respuestas más deterministas
        top_p       = 0.9,
    )
    latency = time.time() - t0
    explanation = completion.choices[0].message.content.strip()

    result = {
        'timestamp'  : str(ts),
        'pred_mw'    : round(pred_mw, 1),
        'real_mw'    : round(real_mw, 1),
        'error_pct'  : round(error, 2),
        'confianza'  : round(1 - error/100, 3),
        'top5_shap'  : top5_shap,
        'explicacion': explanation,
        'latency_s'  : round(latency, 2),
        'llm_model'  : LLM_MODEL,
    }

    if verbose:
        print(f'\n{"="*60}')
        print(f'  TIMESTAMP : {ts}')
        print(f'  PREDICCIÓN: {pred_mw:,.0f} MW')
        print(f'  REAL      : {real_mw:,.0f} MW  (error: {error:.1f}%)')
        print(f'  LATENCIA  : {latency:.2f}s')
        print(f'\n  TOP 5 SHAP:')
        for feat, val in top5_shap.items():
            print(f'    {feat:25s}: {val:+.1f} MW')
        print(f'\n  💬 EXPLICACIÓN LLM:')
        print(f'  {explanation}')
        print('='*60)

    return result


print('✅ Función explain_prediction lista')

## 4. Ejemplos de predicciones explicadas

In [ ]:
# ── Ejemplo 1: hora pico ──────────────────────────────────────────────────────
# Buscar una observación de hora pico (17-18h, día de semana)
idx_pico = X_test[(X_test['hour'].isin([17, 18])) & (X_test['is_weekend'] == 0)].index[0]
idx_num  = X_test.index.get_loc(idx_pico)

print('⏳ Generando explicación para hora pico...')
result_pico = explain_prediction(idx_num)

In [ ]:
# ── Ejemplo 2: hora valle (madrugada, fin de semana) ──────────────────────────
idx_valle = X_test[(X_test['hour'].isin([3, 4])) & (X_test['is_weekend'] == 1)].index[0]
idx_num2  = X_test.index.get_loc(idx_valle)

print('⏳ Generando explicación para hora valle...')
result_valle = explain_prediction(idx_num2)

In [ ]:
# ── Ejemplo 3: predicción con error alto (análisis de errores) ────────────────
# Encontrar predicciones con mayor error
y_pred_all = xgb_model.predict(X_test[FEATURE_COLS])
errors = np.abs(y_pred_all - y_test.values) / y_test.values * 100
idx_alto_error = np.argsort(errors)[-5][()]

print(f'⏳ Generando explicación para predicción con error alto (error: {errors[idx_alto_error]:.1f}%)...')
result_error = explain_prediction(idx_alto_error)

## 5. Evaluación de la calidad del LLM (batch)

In [ ]:
# ── Batch de 10 explicaciones para evaluar consistencia ───────────────────────
print('⏳ Generando batch de 10 explicaciones para evaluación...')
print('   (Esto puede tardar ~15 segundos por límites de rate de Groq)\n')

sample_indices = np.random.RandomState(42).choice(len(X_test), 10, replace=False)
batch_results = []

for i, idx in enumerate(sample_indices):
    print(f'  [{i+1}/10] idx={idx}', end=' ')
    try:
        res = explain_prediction(int(idx), verbose=False)
        batch_results.append(res)
        print(f'→ {res["pred_mw"]:,.0f} MW | error: {res["error_pct"]:.1f}% | latencia: {res["latency_s"]}s')
        time.sleep(1)   # respetar rate limit de Groq
    except Exception as e:
        print(f'⚠️  Error: {e}')

print(f'\n✅ Batch completo: {len(batch_results)}/10 exitosos')

if batch_results:
    avg_latency = np.mean([r['latency_s'] for r in batch_results])
    avg_error   = np.mean([r['error_pct'] for r in batch_results])
    print(f'   Latencia promedio LLM: {avg_latency:.2f}s')
    print(f'   Error promedio modelo: {avg_error:.2f}%')

# Guardar resultados
with open('data/processed/llm_batch_results.json', 'w', encoding='utf-8') as f:
    json.dump(batch_results, f, ensure_ascii=False, indent=2)
print('\n💾 Resultados guardados en data/processed/llm_batch_results.json')

## 6. Interfaz de consulta interactiva

In [ ]:
# ── Consulta personalizada ─────────────────────────────────────────────────────
# Puedes cambiar el índice para explorar diferentes predicciones

def consultar(hora=17, dia_semana='lunes', mes='enero'):
    """
    Interfaz amigable: busca una observación con los criterios dados
    y genera la explicación LLM.
    """
    dow_map = {'lunes':0,'martes':1,'miercoles':2,'jueves':3,'viernes':4,'sabado':5,'domingo':6}
    mes_map = {'enero':1,'febrero':2,'marzo':3,'abril':4,'mayo':5,'junio':6,
               'julio':7,'agosto':8,'septiembre':9,'octubre':10,'noviembre':11,'diciembre':12}

    dow_n = dow_map.get(dia_semana.lower(), 0)
    mes_n = mes_map.get(mes.lower(), 1)

    mask = ((X_test['hour'] == hora) &
            (X_test['dayofweek'] == dow_n) &
            (X_test['month'] == mes_n))

    if mask.sum() == 0:
        print(f'⚠️  No se encontraron observaciones para {hora}h, {dia_semana} de {mes}')
        return

    idx = X_test[mask].index[0]
    idx_num = X_test.index.get_loc(idx)
    print(f'🔍 Encontrada observación: {idx}')
    return explain_prediction(idx_num)


# Ejemplo de uso:
print('Consultando: lunes 18h de enero...')
res = consultar(hora=18, dia_semana='lunes', mes='enero')

In [ ]:
# ── Resumen final del componente LLM ─────────────────────────────────────────
print('='*60)
print('  RESUMEN — COMPONENTE LLM')
print('='*60)
print(f'  Modelo LLM:        {LLM_MODEL} vía Groq API')
print(f'  Estrategia:        Few-shot (2 ejemplos) + chain-of-thought')
print(f'  Rol del LLM:       Explicabilidad de predicciones ML')
print(f'  Inputs al LLM:     pred_mw, timestamp, top5 SHAP values')
print(f'  Output:            Explicación en español (≤4 oraciones)')
print(f'  Latencia promedio: ~1.5s por llamada')
print(f'  Costo:             Gratuito (tier gratuito Groq)')
print('='*60)
print('\n✅ Pipeline completo: EDA → Preprocesamiento → Modelado → LLM')
print('   → Continúa con app/main.py para la demo Streamlit (opcional)')